# Chapitre 5 — Nettoyage des données

## 5.2 Gestion des valeurs manquantes

---

### Objectifs

À la fin de cette leçon, vous serez capable de :
- **Identifier** les valeurs manquantes dans un DataFrame
- **Comprendre** pourquoi les données sont manquantes (MCAR, MAR, MNAR)
- **Décider** quand supprimer des lignes ou des colonnes
- **Appliquer** les stratégies de suppression appropriées avec pandas

---

## 5.2.1 Identifier les valeurs manquantes

Avant de traiter les valeurs manquantes, il faut les **trouver** et **quantifier** leur importance.

In [ ]:
import pandas as pd
import numpy as np

# Créer un DataFrame de démonstration avec des valeurs manquantes
np.random.seed(42)
df = pd.DataFrame({
    'id': range(100),
    'nom': [f'Client_{i}' if np.random.random() > 0.05 else None for i in range(100)],
    'email': [f'email_{i}@test.com' if np.random.random() > 0.02 else None for i in range(100)],
    'age': [np.random.randint(18, 70) if np.random.random() > 0.08 else None for i in range(100)],
    'revenu': [np.random.randint(20000, 100000) if np.random.random() > 0.25 else None for i in range(100)],
    'fax': [f'Fax_{i}' if np.random.random() > 0.95 else None for i in range(100)]  # 95% missing
})

print("DataFrame créé :")
print(df.head(10))

In [ ]:
# Méthode 1 : Compter les valeurs manquantes par colonne
print("Nombre de valeurs manquantes par colonne :")
print(df.isnull().sum())

In [ ]:
# Méthode 2 : Pourcentage de valeurs manquantes
print("Pourcentage de valeurs manquantes par colonne :")
missing_pct = (df.isnull().mean() * 100).round(1)
print(missing_pct)

In [ ]:
# Méthode 3 : Vue d'ensemble avec info()
print("\nInfo du DataFrame :")
df.info()

In [ ]:
# Méthode 4 : Visualisation avec heatmap
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.imshow(df.isnull(), aspect='auto', cmap='Reds')
plt.colorbar(label='Manquant (1) / Présent (0)')
plt.xlabel('Colonnes')
plt.ylabel('Lignes')
plt.title('Carte des valeurs manquantes')
plt.xticks(range(len(df.columns)), df.columns, rotation=45)
plt.tight_layout()
plt.show()

---

## 5.2.2 Comprendre POURQUOI les données sont manquantes

Avant de décider quoi faire, il est crucial de comprendre **pourquoi** les données manquent. Il existe trois types :

```
┌─────────────────────────────────────────────────────────────────────┐
│              TYPES DE DONNÉES MANQUANTES                            │
├─────────────────────┬───────────────────────────────────────────────┤
│        MCAR         │  Missing Completely At Random                 │
│                     │  Le manque est TOTALEMENT aléatoire           │
│                     │  Ex: Erreur de saisie, bug système            │
│                     │  → Suppression généralement OK                │
├─────────────────────┼───────────────────────────────────────────────┤
│        MAR          │  Missing At Random                            │
│                     │  Le manque dépend d'AUTRES variables          │
│                     │  Ex: Les jeunes répondent moins au sondage    │
│                     │  → Suppression peut créer un biais            │
├─────────────────────┼───────────────────────────────────────────────┤
│        MNAR         │  Missing Not At Random                        │
│                     │  Le manque dépend de la VALEUR elle-même      │
│                     │  Ex: Les riches cachent leur revenu           │
│                     │  → Suppression crée un GROS biais             │
└─────────────────────┴───────────────────────────────────────────────┘
```

**Question :** Si 30% des clients n'ont pas renseigné leur revenu, et que vous supprimez ces lignes, quel biais risquez-vous d'introduire ?

*(Réponse attendue : Si les hauts revenus refusent de déclarer (MNAR), vous sous-estimez le revenu moyen de vos clients)*

---

## 5.2.3 Stratégie 1 : Supprimer des LIGNES

La suppression de lignes est appropriée quand :
- Le pourcentage de missing est **faible** (< 5%)
- Le manque est **MCAR** (aléatoire)
- La colonne concernée est **critique** (ex: email pour un CRM)

| Situation | Recommandation |
|-----------|----------------|
| < 5% missing, MCAR | ✅ Supprimer les lignes |
| 5-20% missing | ⚠️ Réfléchir au biais potentiel |
| > 20% missing | ❌ Ne pas supprimer les lignes |

In [ ]:
# dropna() : Supprimer les lignes avec AU MOINS UNE valeur manquante
# ⚠️ Attention : très destructif !
df_clean = df.dropna()
print(f"Avant : {len(df)} lignes")
print(f"Après dropna() : {len(df_clean)} lignes")
print(f"Lignes supprimées : {len(df) - len(df_clean)} ({(len(df) - len(df_clean))/len(df)*100:.1f}%)")

In [ ]:
# dropna(subset=[...]) : Supprimer seulement si CERTAINES colonnes sont manquantes
# ✅ Beaucoup plus ciblé et moins destructif !

df_clean_subset = df.dropna(subset=['email', 'nom'])
print(f"\nAvant : {len(df)} lignes")
print(f"Après dropna(subset=['email', 'nom']) : {len(df_clean_subset)} lignes")
print("→ On garde les lignes même si 'revenu' ou 'fax' sont manquants")

In [ ]:
# dropna(how='all') : Supprimer seulement si TOUTES les valeurs sont manquantes
df_clean_all = df.dropna(how='all')
print(f"\nAprès dropna(how='all') : {len(df_clean_all)} lignes")
print("→ Garde les lignes qui ont au moins une valeur non-nulle")

---

## 5.2.4 Stratégie 2 : Supprimer des COLONNES

La suppression de colonnes est appropriée quand :
- Le pourcentage de missing est **très élevé** (> 50-60%)
- La colonne n'est **pas critique** pour l'analyse
- La colonne contient peu d'**information utile**

In [ ]:
# Voir le pourcentage de missing par colonne
missing_pct = (df.isnull().mean() * 100).round(1)
print("Pourcentage de missing par colonne :")
print(missing_pct)
print("\n→ 'fax' a ~95% de missing : candidat à la suppression !")

In [ ]:
# Supprimer une colonne spécifique
df_sans_fax = df.drop(columns=['fax'])
print(f"Colonnes avant : {list(df.columns)}")
print(f"Colonnes après : {list(df_sans_fax.columns)}")

In [ ]:
# Supprimer automatiquement les colonnes avec plus de X% de missing
seuil = 0.5  # 50%
colonnes_a_garder = df.columns[df.isnull().mean() < seuil]
df_clean_cols = df[colonnes_a_garder]

print(f"\nColonnes avant : {list(df.columns)}")
print(f"Colonnes après (< 50% missing) : {list(df_clean_cols.columns)}")

In [ ]:
# Alternative : dropna avec thresh (nombre minimum de valeurs non-nulles)
seuil_valeurs = len(df) * 0.5  # Au moins 50% de valeurs présentes
df_clean_thresh = df.dropna(axis=1, thresh=int(seuil_valeurs))

print(f"\nAvec dropna(axis=1, thresh={int(seuil_valeurs)}) :")
print(f"Colonnes gardées : {list(df_clean_thresh.columns)}")

---

## 5.2.5 Arbre de décision

```
                    VALEURS MANQUANTES DÉTECTÉES
                              │
                              ▼
                 Quel % de missing dans la COLONNE ?
                              │
              ┌───────────────┼───────────────┐
              ▼               ▼               ▼
          > 50-60%        5% - 50%         < 5%
              │               │               │
              ▼               ▼               ▼
      Colonne critique?   Garder les      MCAR ?
              │            NaN pour          │
        ┌─────┴─────┐       ML          ┌────┴────┐
        ▼           ▼       │           ▼         ▼
       NON         OUI      │          OUI       NON
        │           │       │           │         │
        ▼           ▼       │           ▼         ▼
   SUPPRIMER    Garder      │      SUPPRIMER   Garder
   la colonne   pour ML     │      les lignes  pour ML
                            │
                            ▼
              ┌─────────────────────────────────┐
              │  Les NaN restants seront        │
              │  traités dans le Pipeline ML    │
              │  (Module 3 - SimpleImputer)     │
              └─────────────────────────────────┘
```

---

## ✍️ Exercice 5.2 : Diagnostic et décision

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
df_ex = pd.DataFrame({
    'client_id': range(1000),
    'nom': ['Client_' + str(i) for i in range(1000)],
    'age': np.where(np.random.random(1000) < 0.03, np.nan, np.random.randint(18, 70, 1000)),
    'email': [np.nan if np.random.random() < 0.02 else f'email_{i}@test.com' for i in range(1000)],
    'revenu': np.where(np.random.random(1000) < 0.25, np.nan, np.random.randint(20000, 100000, 1000)),
    'fax': [np.nan if np.random.random() < 0.95 else f'fax_{i}' for i in range(1000)]
})

print("Pourcentage de missing par colonne :")
print((df_ex.isnull().mean() * 100).round(1))

In [ ]:
# Analysez et décidez pour chaque colonne :
# 
# client_id : 0% → _____
# nom : 0% → _____
# age (~3%) : _____ 
# email (~2%) : _____
# revenu (~25%) : _____
# fax (~95%) : _____

In [ ]:
# Solution
print("=== DÉCISIONS ===")
print()
print("• client_id (0%) : Pas d'action - clé primaire")
print("• nom (0%) : Pas d'action")
print("• age (~3%) : MCAR probable → dropna(subset=['age']) OU garder NaN pour ML")
print("• email (~2%) : Critique pour contact → dropna(subset=['email'])")
print("• revenu (~25%) : Trop pour supprimer → Garder les NaN pour ML")
print("• fax (~95%) : Inutile → SUPPRIMER LA COLONNE")

# Appliquer les décisions
df_clean = df_ex.copy()

# 1. Supprimer la colonne fax (>50% missing, non critique)
df_clean = df_clean.drop(columns=['fax'])

# 2. Supprimer les lignes sans email (critique, <5%)
df_clean = df_clean.dropna(subset=['email'])

print(f"\nAvant : {len(df_ex)} lignes, {len(df_ex.columns)} colonnes")
print(f"Après : {len(df_clean)} lignes, {len(df_clean.columns)} colonnes")
print(f"\nNaN restants (seront traités en Module 3) :")
print(df_clean.isnull().sum())

---

## 📝 Résumé

| Méthode | Syntaxe | Quand l'utiliser |
|---------|---------|------------------|
| Supprimer lignes (toutes) | `df.dropna()` | Rarement - trop destructif |
| Supprimer lignes (ciblé) | `df.dropna(subset=['col'])` | Colonnes critiques, < 5% |
| Supprimer colonnes | `df.drop(columns=['col'])` | > 50% missing, non critique |
| Supprimer colonnes (seuil) | `df.dropna(axis=1, thresh=n)` | Automatiser avec seuil |

---

## ➡️ Et les valeurs manquantes restantes ?

Après le nettoyage, votre DataFrame contient encore des NaN (comme `age` et `revenu` dans notre exemple). **C'est normal !**

**Ces NaN seront traités dans le Module 3 (Machine Learning)** avec `SimpleImputer` dans un Pipeline sklearn.

```
┌─────────────────────────────────────────────────────────────────────┐
│  Module 2 (Nettoyage)       │    Module 3 (ML Pipeline)             │
├─────────────────────────────┼───────────────────────────────────────┤
│  • Supprimer colonnes       │    • SimpleImputer(strategy='median') │
│    avec >50% missing        │    • Fit sur train uniquement         │
│  • Supprimer lignes si      │    • Transform sur train et test      │
│    colonne critique <5%     │    • Pas de data leakage !            │
│  • Garder les autres NaN    │                                       │
└─────────────────────────────┴───────────────────────────────────────┘
```

> 💡 **Pourquoi ne pas imputer maintenant ?** Si vous calculez la moyenne sur tout le dataset pour imputer, vous incluez les données de test → **data leakage**. Le Module 3 vous montrera comment faire correctement.